# Giải trình phương pháp chấm điểm chuyển đổi số ngân hàng

Notebook này giải thích và kiểm tra quy trình chấm 5 ngân hàng × 16 tiêu chí. **Điểm gợi ý không phải điểm chính thức**: chỉ dùng kết quả từ `scores.csv` và `ranking.csv` khi bảng `review_scores.xlsx` đã được nhóm duyệt đầy đủ bằng `python3 scoring.py score`.

## 1. Nguồn dữ liệu

- `data/PCCV-NHS-2.xlsx`, sheet `task1-Khung 16 chỉ tiêu`: danh sách tiêu chí và rubric 0–1–2 gốc.
- Ưu tiên `../Digital-banking-scoring-system/Evidence_Final.xlsx`, sheet `Evidence_Raw`, lấy trực tiếp từ repository GitHub đã clone; nếu không có thì dùng `data/evidence.xlsx` làm bản dự phòng. Workbook upstream hiện có cùng 2.708 dòng với bản evidence cục bộ. Tier 2 là tín hiệu lọc, **không phải điểm rubric**.
- Các báo cáo thường niên/bền vững 2025 được tham chiếu trong cột `filename` của evidence. Cần đối chiếu bản báo cáo gốc trước khi duyệt điểm.
- `output/review_scores.xlsx`: bảng 80 trường hợp để nhóm xác minh, ghi điểm, người duyệt và lý do.

## 2. Chuẩn hóa dữ liệu

Tên ngân hàng được ánh xạ về BIDV, MB, OCB, VCB, VietinBank; mã tiêu chí chuyển thành chữ hoa; văn bản bằng chứng chuyển chữ thường, bỏ dấu tiếng Việt và chuẩn hóa khoảng trắng để tìm tín hiệu. Dòng thiếu ngân hàng/tiêu chí hợp lệ, trang hoặc ngữ cảnh sẽ được tách sang `invalid_evidence.csv`; dòng trùng được loại theo ngân hàng, tiêu chí, nguồn, trang, khớp từ khóa, ngữ cảnh và tầng lọc.

## 3. Quy tắc chấm điểm

Với mỗi cặp ngân hàng–tiêu chí, người duyệt đối chiếu **đoạn bằng chứng cùng ngữ cảnh** với rubric gốc:

- **0**: chưa tìm thấy bằng chứng đạt mức 1 trong phạm vi nguồn đã rà soát; không được hiểu là khẳng định ngân hàng không triển khai.
- **1**: có bằng chứng cho hoạt động/sản phẩm/chủ đề ở mức cơ bản.
- **2**: chỉ khi bằng chứng còn thể hiện rõ điều kiện nâng mức cụ thể của rubric (quy mô, độ phủ, tích hợp, nền tảng, cơ chế, nhiều nghiệp vụ/quy trình…). Hai nhóm từ khóa cùng xuất hiện không tự động đủ điều kiện.

Chương trình tạo `suggested_score` theo tín hiệu rubric; `rubric_match`, `reason`, `warning`, `confidence`, báo cáo và trang hỗ trợ diễn giải. Trạng thái `automated` biểu thị điểm do thuật toán tạo, không phải được con người xác minh. Kết quả cần được gọi là **BDTI ước tính từ phương pháp chấm điểm tự động**. Các điểm 2 và điểm tin cậy thấp cần được lưu ý khi diễn giải. VCB–D1 không có candidate trong `Evidence_Raw`, nên điểm 0 chỉ phản ánh chưa tìm thấy bằng chứng trong tập dữ liệu, không chứng minh ngân hàng không triển khai.

## 4. Công thức A–D và BDTI

Mỗi trụ cột A, B, C, D có 4 tiêu chí, tối đa 8 điểm. Điểm trụ cột chuẩn hóa về thang 100:

`Điểm trụ cột = tổng điểm 4 tiêu chí / 8 × 100`

`BDTI = (A + B + C + D) / 4 = tổng điểm 16 tiêu chí / 32 × 100`

Các trụ cột được cho trọng số bằng nhau. Thứ hạng áp dụng **competition ranking**: đồng hạng được giữ nguyên (ví dụ 1, 1, 3, 3, 5). `display_order` chỉ là thứ tự hiển thị; không phá đồng hạng.

## 5. Kiểm tra và kết quả

Các cell dưới đây kiểm tra tính đầy đủ của 16 tiêu chí/80 cặp, miền điểm, đủ 4 tiêu chí mỗi trụ cột và đối chiếu BDTI theo hai công thức. Notebook dùng output của pipeline tự động và luôn giữ nhãn phương pháp, trạng thái, cảnh báo và độ tin cậy; không diễn giải kết quả này thành đánh giá đã được con người xác minh.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scoring.py").is_file()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("Không tìm thấy scoring.py trong thư mục hiện tại hoặc thư mục cha.")

DATA = PROJECT / "data"
OUT = PROJECT / "output"
BANKS = ["BIDV", "MB", "OCB", "VCB", "VietinBank"]
CODES = [f"{pillar}{i}" for pillar in "ABCD" for i in range(1, 5)]
print(f"Thư mục dự án: {PROJECT}")

Thư mục dự án: /Users/thylnh.iu/Documents/Ngân hàng số/BANK_DX_PROJECT


In [2]:
framework = pd.read_excel(DATA / "PCCV-NHS-2.xlsx", sheet_name="task1-Khung 16 chỉ tiêu")
upstream_evidence = PROJECT.parent / "Digital-banking-scoring-system" / "Evidence_Final.xlsx"
evidence_path = upstream_evidence if upstream_evidence.is_file() else DATA / "evidence.xlsx"
evidence = pd.read_excel(evidence_path, sheet_name="Evidence_Raw")
review = pd.read_excel(OUT / "review_scores.xlsx", sheet_name="Scoring_Review")

print("Nguồn evidence:", evidence_path)
print("Rubric:", len(framework), "dòng; Evidence_Raw:", len(evidence), "dòng; Review:", len(review), "cặp")
print("Nguồn báo cáo được tham chiếu:")
display(pd.DataFrame({"report": sorted(evidence["filename"].dropna().astype(str).unique())}))

assert len(framework) == 16 and framework["Mã"].astype(str).str.strip().str.upper().nunique() == 16
assert len(review) == 80 and not review.duplicated(["bank", "criterion"]).any()
assert set(review["bank"]) == set(BANKS)
assert set(review["criterion"]) == set(CODES)
print("ĐẠT: 16 tiêu chí và đủ 80 cặp ngân hàng–tiêu chí.")

Nguồn evidence: /Users/thylnh.iu/Documents/Ngân hàng số/Digital-banking-scoring-system/Evidence_Final.xlsx
Rubric: 16 dòng; Evidence_Raw: 2708 dòng; Review: 80 cặp
Nguồn báo cáo được tham chiếu:


,report
0,BIDV_AnnualReport_2025.pdf
1,BIDV_SustainabilityReport_2025.pdf
2,MB_AnnualReport_2025.pdf
3,OCB_AnnualReport_2025.pdf
4,OCB_SustainabilityReport_2025.pdf
5,VCB_AnnualReport_2025.pdf
6,VCB_SustainabilityReport_2025.pdf
7,VietinBank_AnnualReport_2025.pdf
8,VietinBank_SustainabilityReport_2025.pdf


ĐẠT: 16 tiêu chí và đủ 80 cặp ngân hàng–tiêu chí.


### Tiến độ rà soát và điểm gợi ý

Bảng dưới đây là công cụ kiểm soát, không thay cho phán đoán của nhóm. Kiểm tra riêng các điểm 2, các điểm 0, mọi trường hợp thiếu báo cáo/trang và các trường hợp rubric chưa được thể hiện rõ trong đoạn trích.

In [3]:
review["suggested_score"] = pd.to_numeric(review["suggested_score"], errors="coerce")
review["candidate_count"] = pd.to_numeric(review["candidate_count"], errors="coerce").fillna(0).astype(int)
print("Trạng thái duyệt:")
display(review["status"].fillna("(trống)").value_counts().rename_axis("status").to_frame("cases"))
print("Điểm gợi ý:")
display(review["suggested_score"].value_counts(dropna=False).sort_index().rename_axis("score").to_frame("cases"))

priority = review[
    review["suggested_score"].isin([0, 2])
    | review["filename"].isna()
    | review["page"].isna()
    | review["candidate_count"].eq(0)
][["bank", "criterion", "suggested_score", "candidate_count", "filename", "page", "reason", "rubric_match", "review_priority"]]
display(priority.sort_values(["suggested_score", "bank", "criterion"], na_position="first"))

vcb_d1 = review[(review["bank"] == "VCB") & (review["criterion"] == "D1")]
assert len(vcb_d1) == 1
if int(vcb_d1.iloc[0]["candidate_count"]) == 0:
    print("CẢNH BÁO: VCB–D1 không có candidate trong Evidence_Raw; cần rà báo cáo gốc, không tự động kết luận điểm 0.")

Trạng thái duyệt:


,cases
status,
automated,80


Điểm gợi ý:


,cases
score,
0,8
1,40
2,32


,bank,criterion,suggested_score,candidate_count,filename,page,reason,rubric_match,review_priority
7,BIDV,B4,0,15,BIDV_AnnualReport_2025.pdf,8,De xuat 0: doan trich khong co tin hieu muc 1 ...,Muc 1: khong thay; Muc 2: khong thay,Cao: xac minh bao cao goc va pham vi tim kiem
23,MB,B4,0,21,MB_AnnualReport_2025.pdf,9,De xuat 0: doan trich khong co tin hieu muc 1 ...,Muc 1: khong thay; Muc 2: khong thay,Cao: xac minh bao cao goc va pham vi tim kiem
55,VCB,B4,0,5,VCB_SustainabilityReport_2025.pdf,8,De xuat 0: doan trich khong co tin hieu muc 1 ...,Muc 1: khong thay; Muc 2: khong thay,Cao: xac minh bao cao goc va pham vi tim kiem
58,VCB,C3,0,1,VCB_SustainabilityReport_2025.pdf,52,De xuat 0: doan trich khong co tin hieu muc 1 ...,Muc 1: khong thay; Muc 2: khong thay,Cao: xac minh bao cao goc va pham vi tim kiem
60,VCB,D1,0,0,VCB_AnnualReport_2025.pdf; VCB_SustainabilityR...,N/A - khong co candidate; can ra soat bao cao,Khong co candidate trong Evidence_Raw; khong c...,NaN,Cao: ra soat thu cong bao cao goc; chua co tra...
61,VCB,D2,0,10,VCB_SustainabilityReport_2025.pdf,11,De xuat 0: doan trich khong co tin hieu muc 1 ...,Muc 1: khong thay; Muc 2: khong thay,Cao: xac minh bao cao goc va pham vi tim kiem
63,VCB,D4,0,2,VCB_AnnualReport_2025.pdf,33,De xuat 0: doan trich khong co tin hieu muc 1 ...,Muc 1: khong thay; Muc 2: khong thay,Cao: xac minh bao cao goc va pham vi tim kiem
66,VietinBank,A3,0,8,VietinBank_AnnualReport_2025.pdf,3,De xuat 0: doan trich khong co tin hieu muc 1 ...,Muc 1: khong thay; Muc 2: khong thay,Cao: xac minh bao cao goc va pham vi tim kiem
3,BIDV,A4,2,41,BIDV_AnnualReport_2025.pdf,43,De xuat 2: doan bang chung co tin hieu muc 1 (...,Muc 1: dao tao; Muc 2: chuong trinh dao tao,Cao: xac minh dieu kien muc 2 trong ngu canh
4,BIDV,B1,2,27,BIDV_SustainabilityReport_2025.pdf,36,De xuat 2: doan bang chung co tin hieu muc 1 (...,Muc 1: ekyc; Muc 2: tich hop,Cao: xac minh dieu kien muc 2 trong ngu canh


CẢNH BÁO: VCB–D1 không có candidate trong Evidence_Raw; cần rà báo cáo gốc, không tự động kết luận điểm 0.


## 6. Tính lại và xác minh kết quả

Các file `processed_scores.csv`, `scores.csv`, `ranking.csv` là bộ dữ liệu BDTI ước tính từ phương pháp chấm điểm tự động. `status=automated` và cột `methodology` mang nhãn này để dashboard hiển thị minh bạch.

In [4]:
automated_files = [OUT / name for name in ["processed_scores.csv", "scores.csv", "ranking.csv"]]
if all(path.is_file() for path in automated_files):
    details = pd.read_csv(automated_files[0])
    scores = pd.read_csv(automated_files[1])
    ranking = pd.read_csv(automated_files[2])
    source_label = "BDTI ƯỚC TÍNH – phương pháp chấm điểm tự động"
    expected = review[["bank", "criterion", "suggested_score"]].copy()
    expected["suggested_score"] = pd.to_numeric(expected["suggested_score"], errors="coerce")
    actual = details[["bank", "criterion", "final_score"]].copy()
    actual["final_score"] = pd.to_numeric(actual["final_score"], errors="coerce")
    compared = expected.merge(actual, on=["bank", "criterion"], validate="one_to_one")
    assert compared["suggested_score"].equals(compared["final_score"]), "Output chính thức không khớp bảng review hiện tại."
else:
    provisional_detail = OUT / "provisional_detail.csv"
    provisional_scores = OUT / "provisional_scores.csv"
    provisional_ranking = OUT / "provisional_ranking.csv"
    if not all(p.is_file() for p in [provisional_detail, provisional_scores, provisional_ranking]):
        raise FileNotFoundError("Chưa có output; chạy python3 scoring.py all để tạo dữ liệu sơ bộ.")
    details = pd.read_csv(provisional_detail)
    scores = pd.read_csv(provisional_scores)
    ranking = pd.read_csv(provisional_ranking)
    source_label = "BDTI ƯỚC TÍNH SƠ BỘ – phương pháp chấm điểm tự động"

print(source_label)
assert len(details) == 80 and not details.duplicated(["bank", "criterion"]).any()
assert pd.to_numeric(details["final_score"], errors="coerce").isin([0, 1, 2]).all()
pillar_counts = details.assign(pillar=details["criterion"].astype(str).str[0]).groupby(["bank", "pillar"])["criterion"].nunique()
assert len(pillar_counts) == 20 and pillar_counts.eq(4).all()

manual_bdti = details.groupby("bank")["final_score"].sum().div(32).mul(100).sort_index()
published_bdti = scores.set_index("bank")["BDTI"].sort_index()
assert np.allclose(manual_bdti.loc[BANKS], published_bdti.loc[BANKS])
assert ranking["rank"].is_monotonic_increasing
assert details["status"].eq("automated").all()
assert scores["status"].eq("automated").all()
assert ranking["status"].eq("automated").all()
print("ĐẠT: 80 điểm hợp lệ, công thức BDTI khớp; trạng thái và nhãn phương pháp đều là automated.")
display(ranking[["rank", "bank", "BDTI", "A", "B", "C", "D", "status", "methodology", "warning_count", "low_confidence_count"]])

BDTI ƯỚC TÍNH – phương pháp chấm điểm tự động
ĐẠT: 80 điểm hợp lệ, công thức BDTI khớp; trạng thái và nhãn phương pháp đều là automated.


,rank,bank,BDTI,A,B,C,D,status,methodology,warning_count,low_confidence_count
0,1,OCB,78.125,62.5,87.5,87.5,75.0,automated,BDTI ước tính từ phương pháp chấm điểm tự động,9,0
1,2,MB,71.875,75.0,50.0,100.0,62.5,automated,BDTI ước tính từ phương pháp chấm điểm tự động,9,1
2,3,BIDV,65.625,62.5,50.0,75.0,75.0,automated,BDTI ước tính từ phương pháp chấm điểm tự động,7,1
3,4,VietinBank,59.375,37.5,62.5,62.5,75.0,automated,BDTI ước tính từ phương pháp chấm điểm tự động,5,1
4,5,VCB,50.000,62.5,62.5,50.0,25.0,automated,BDTI ước tính từ phương pháp chấm điểm tự động,10,5


## 7. Hạn chế của phương pháp

1. Tìm kiếm từ khóa và quy tắc ngữ cảnh chỉ tạo gợi ý; không hiểu đầy đủ phủ định, phạm vi áp dụng, tính trọng yếu, thời điểm hoặc chất lượng triển khai như người đọc báo cáo.
2. Đoạn trích có thể bị cắt, OCR sai, hoặc thiếu ngữ cảnh; không tìm thấy candidate không chứng minh rằng ngân hàng không có hoạt động đó.
3. Báo cáo tự công bố có khác biệt về độ dài, mức chi tiết và kỳ báo cáo; số trang/nguồn cần được người duyệt đối chiếu bản gốc.
4. Trọng số bằng nhau giữa 4 trụ cột và giữa 16 tiêu chí là giả định của chỉ số; kết quả phụ thuộc rubric và không đo trực tiếp tác động/hiệu quả chuyển đổi số.
5. Điểm số là thang thứ bậc 0–1–2 nhưng phép cộng coi khoảng cách giữa các mức là bằng nhau; BDTI là chỉ số tổng hợp để so sánh, không phải xác suất hay kết luận kiểm toán.
6. Đồng hạng được giữ theo competition ranking; không áp dụng tiêu chí phụ. Chênh lệch nhỏ/đồng hạng không nên được diễn giải là khác biệt thực chất.

Trước khi chuyển sang dashboard, cần hoàn thành phê duyệt 80 dòng, điền người duyệt/lý do/trang báo cáo, xử lý VCB–D1 bằng rà soát nguồn, rồi chạy `python3 scoring.py score` và chạy lại notebook để nhúng kết quả chính thức.